# Etap 1: Przygotowanie danych do prognozy sprzedażowej

Projekt opiera się na rzeczywistym, 5-letnim zbiorze danych (2021-2025) pochodzącym z wewnętrznego systemu hurtowni mięs i wędlin. Dane obejmują sprzedaż B2B dla dwóch głównych segmentów: sklepów detalicznych oraz klientów z sektora HoReCa (restauracje).

Poniższy notebook jest poświęcony przetwarzaniu danych (ETL) – przygotowuje surowe dane transakcyjne pod algorytmy uczenia maszynowego dla klientów gastronomicznych (HoReCa).

**Rekategoryzacja asortymentu**
Ze względu na wysoką ogólnikowość oryginalnych grup towarowych (spośród 3029 unikalnych produktów aż 2219 było przypisanych do ogólnej grupy "Wędliny"), przeprowadzono ręczną kategoryzację ekspercką. Przypisanie produktów do nowych kategorii znajduje się w słowniku `ProduktyKategorie.csv` (kolumny `rodzaj_towaru` i `pierwsze_nazwa`).
* *Wynik:* Wyodrębniono **11 głównych kategorii** (m.in. Mięso Świeże, Kiełbasy Cienkie, Wyroby Podrobowe) oraz **53 podkategorie**.

### 1. Integracja danych (Data Merging)
Wczytywane są dwa pliki CSV (transakcje sprzedażowe oraz słownik asortymentu), które następnie łączone są w jedną tabelę za pomocą operacji `LEFT JOIN` po kluczu `ID_artykułu`.

In [ ]:
import numpy as np
import pandas as pd

#Wczytanie plików
df1 = pd.read_csv(r'../dane/surowe/Sprzedaz.csv',sep=';', encoding='utf-8-sig')  # sprzedaż
df2 = pd.read_csv(r'../dane/surowe/ProduktyKategorie.csv',sep=';', encoding='utf-8-sig')  # dane artykułów

# Łączenie po kolumnie ID_artykułu
df_merged = pd.merge(df1, df2, on="ID_artykułu", how="left") 

sales = df_merged
sales.head()

,Klient,data_sprzedaży,ID_artykułu,Cena_zakupu,Cena_sprzedaży,Ilość,Wartość_netto,Stawka_VAT,Nazwa,grupa_towarów,producent,rodzaj_towarów,pierwsze_nazwa,rodzaj_towaru
0,S2,4.01.2021,529,"19,52","22,49","4,84","108,85",5%,Boczek wędzony extra <>,Zygmunt,Zygmunt,Wędliny,Boczki i Bekony,4. Wędzonki i Boczki
1,S2,4.01.2021,7376,"19,05","21,99","5,06","111,27",5%,Kiełbasa góralska,Zygmunt,Zygmunt,Wędliny,Kiełbasy tradycyjne i regionalne,1. Kiełbasy Cienkie
2,S2,4.01.2021,6,"17,8","19,99","2,12","42,38",5%,Kiełbasa szynkowa wp.extra,ALEKSANDRIA,Aleksandria,Wędliny,Szynkowe i Parzone,2. Kiełbasy Grube i Suche
3,S2,4.01.2021,6941,1,"1,29","5,5","7,1",5%,Porcja z kurczaka,DRÓB,Yxx,DRÓB,Drób,9. Mięso Świeże i Surowe
4,S2,4.01.2021,203,"14,8","16,35","0,92","15,04",5%,Senacka kiełbasa,H.A.M,Ham,H.A.M,Kiełbasy Codzienne i Grillowe,1. Kiełbasy Cienkie


### 2. Czyszczenie i transformacja typów (Data Cleaning & Parsing)
Proces przygotowania danych wymagał ujednolicenia formatów, usunięcia szumu oraz korekty błędów ludzkich i wartości odstających (outlierów). Część błędów poprawiono ręcznie w pliku źródłowym, pozostałe są obsługiwane w kodzie poniżej:

* **Konwersja typów danych:** Daty rzutowano na format `datetime`. Zmienne numeryczne z polskim formatowaniem (przecinki jako separatory dziesiętne) przekonwertowano na kropki i przypisano im typ zmiennoprzecinkowy `float64`. Kolumny z kategoryzacji otrzymały nazwy `Podkategoria` i `Kategoria`.
* **Usunięcie duplikatów:** Wykryto i usunięto 467 całkowicie zduplikowanych wierszy.
* **Selekcja kolumn:** Usunięto zmienne, które nie są wykorzystywane w modelu (stawka VAT, nazwa produktu, producent, grupa towarów, ID artykułu, ceny zakupu i sprzedaży oraz wartość netto). Do dalszych etapów trafiają tylko: klient, data, ilość, podkategoria i kategoria.
* **Usuwanie wartości ujemnych i szumu ilościowego:** Pozostawiono wyłącznie transakcje o ilości powyżej 0,5 kg. Filtr usuwa zarówno błędy systemowe (2 transakcje z ujemną ilością), jak i mikro-zamówienia do 0,5 kg (prawdopodobnie darmowe akcje degustacyjne), które zakłócałyby predykcję hurtową.
* **Korekta błędów ludzkich (ręcznie, w pliku `Sprzedaz.csv`):** Poprawiono skrajnie błędne ceny wynikające z literówek (np. wpisanie 954 zł/kg za polędwicę skorygowano do rynkowej średniej dla tego okresu).
* **Obsługa skrajnych outlierów (ręcznie, w pliku `Sprzedaz.csv`):** Anomalie ilościowe — takie jak zamówienie 1,3 tony indyka przez sklep, którego historyczne maksimum wynosiło 14 kg — zastąpiono średnią wartością zamówień tego klienta, aby nie zaburzać uczenia modelu.
* **Zjawiska promocyjne:** Zidentyfikowano 10 rekordów z ceną zakupu równą 0 (wynik gratisowych akcji od dystrybutora). Uznano je za poprawne zjawisko biznesowe i pozostawiono w danych.

In [ ]:
sales['data_sprzedaży'] = pd.to_datetime(sales['data_sprzedaży'],dayfirst=True, errors='coerce')

sales['ID_artykułu'] = sales['ID_artykułu'].astype('int64')
sales['Cena_zakupu'] = sales['Cena_zakupu'].str.replace(',', '.').astype('float64')
sales['Cena_sprzedaży'] = sales['Cena_sprzedaży'].str.replace(',', '.').astype('float64')
sales['Ilość'] = sales['Ilość'].str.replace(',', '.').astype('float64')
sales['Wartość_netto'] = sales['Wartość_netto'].str.replace(',', '.').astype('float64')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196045 entries, 0 to 196044
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Klient          196045 non-null  object        
 1   data_sprzedaży  196045 non-null  datetime64[ns]
 2   Ilość           196045 non-null  float64       
 3   Podkategoria    196045 non-null  object        
 4   Kategoria       196045 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 53.0 MB


In [ ]:
# Sprawdzenie duplikatów
sales.duplicated().sum()
sales[sales.duplicated(keep=False)]

In [ ]:
sales.drop_duplicates(inplace=True)

In [ ]:
sales = sales.rename(columns={'pierwsze_nazwa': 'Podkategoria'})
sales = sales.rename(columns={'rodzaj_towaru': 'Kategoria'})

sales = sales.drop(columns=['Stawka_VAT','grupa_towarów','rodzaj_towarów','Wartość_netto', 'Nazwa','producent','Cena_sprzedaży','Cena_zakupu','ID_artykułu'])

sales.info(memory_usage='deep')

In [4]:
# Sprawdzenie wartości minusowych
sales[sales['Ilość'] <0]

,Klient,data_sprzedaży,Ilość,Podkategoria,Kategoria
60948,S4,2022-09-07,-1.50,Kaszanki i Krupnioki,7. Wyroby Podrobowe
83267,S2,2023-04-26,-6.24,Galareta,8. Garmażeria i Dania Gotowe


In [ ]:
# Przefiltrowanie i pozbycie się wartości poniżej 0.5 kg
sales = sales[sales['Ilość'] > 0.5]

### 3. Filtrowanie segmentu HoReCa i agregacja
Dane są zawężane wyłącznie do klientów gastronomicznych (restauracje R1-R5). Następnie transakcje są agregowane na poziom dnia, klienta i podkategorii produktu, aby zsumować całkowity dzienny popyt na dany asortyment.

In [10]:
sklepy_lista = ['R1', 'R2', 'R3','R4','R5']
df = sales[sales['Klient'].isin(sklepy_lista)].copy()

In [11]:
# Agregacja przed siatką i kalendarzem 
df = df.groupby(['data_sprzedaży','Klient', 'Podkategoria'])['Ilość'].sum().reset_index()

### 4. Generowanie siatki czasowej (Time-Series Grid)
W oryginalnych danych brakuje wierszy dla dni, w których klient nie zamówił danej podkategorii. Dlatego na podstawie pliku kalendarza (1826 dni) tworzona jest pełna siatka wszystkich kombinacji: data × klient × podkategoria. Z kalendarza usuwane są kolumny pomocnicze (`miesiac`, `kwartał`, `dzien_tygodnia`, `nazwa_swieta`) – model korzysta z ich wersji cyklicznych (sin/cos).

In [12]:
# Wczytanie kalendarza w którym jest pełny zakres dat
kalendarz = pd.read_csv(r'../dane/kalendarz/Kalendarz.csv', sep=';', decimal=',', encoding='utf-8-sig')
kalendarz['data'] = pd.to_datetime(kalendarz['data'], format='%Y-%m-%d')
kalendarz = kalendarz.drop(columns=['miesiac', 'kwartał', 'dzien_tygodnia', 'nazwa_swieta'])

# Stworzenie siatki (wszystkie kombinacje data x klient x podkategoria)

klienci = df['Klient'].unique()
podkategorie = df['Podkategoria'].unique()
daty = kalendarz['data'].unique()

index = pd.MultiIndex.from_product(
    [daty, klienci,podkategorie],
    names=['data_sprzedaży', 'Klient','Podkategoria'])

szablon = pd.DataFrame(index=index).reset_index()

#### 4.1 Połączenie siatki z danymi i kalendarzem oraz uzupełnienie braków zerami
Do siatki dołączana jest zagregowana sprzedaż (`LEFT JOIN`), a następnie cechy kalendarzowe dla każdej daty. Dni bez sprzedaży (`NaN`) uzupełniane są wartością 0, co zapewnia ciągłość szeregu czasowego. Wynik: 465 630 wierszy (1826 dni × 5 restauracji × 51 podkategorii).

In [13]:
# Łączenie siatki z danymi 
df = pd.merge(szablon, df, on=['data_sprzedaży', 'Klient','Podkategoria'], how='left')
print("Suma po siatce i merge:", df['Ilość'].sum())

# Połączenie danych z kalendarzem 

df = df.merge(kalendarz, left_on='data_sprzedaży', right_on='data', how='left')
df.drop(columns=['data'], inplace=True)

# Sprawdzenie i zastąpienie 0 brakujących wartości
df['Ilość'] = df['Ilość'].fillna(0)
print(df.shape)
df.head()

Suma po siatce i merge: 717417.0799999998
(465630, 20)


,data_sprzedaży,Klient,Podkategoria,Ilość,miesiac_sin,miesiac_cos,kwartał_sin,kwartał_cos,day_sin,day_cos,tydzien_roku,rok,czy_niedziela,okres_wielkanocny,okres_wigilijny,czy_majowka,czy_ferie_slask,czy_wakacje,czy_swieto,dni_do_swieta
0,2021-01-01,R3,Boczki i Bekony,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0
1,2021-01-01,R3,Kaszanki i Krupnioki,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0
2,2021-01-01,R3,Szynki Tradycyjne i Wędzone,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0
3,2021-01-01,R3,Wieprzowina,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0
4,2021-01-01,R3,Wołowina i Cielęcina,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0


In [14]:
# Sprawdzenie i dodawanie 0 przy NaN
df['Ilość'] = df['Ilość'].fillna(0)
print(df.shape)
df.head()

(465630, 20)


,data_sprzedaży,Klient,Podkategoria,Ilość,miesiac_sin,miesiac_cos,kwartał_sin,kwartał_cos,day_sin,day_cos,tydzien_roku,rok,czy_niedziela,okres_wielkanocny,okres_wigilijny,czy_majowka,czy_ferie_slask,czy_wakacje,czy_swieto,dni_do_swieta
0,2021-01-01,R3,Boczki i Bekony,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0
1,2021-01-01,R3,Kaszanki i Krupnioki,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0
2,2021-01-01,R3,Szynki Tradycyjne i Wędzone,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0
3,2021-01-01,R3,Wieprzowina,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0
4,2021-01-01,R3,Wołowina i Cielęcina,0.0,0.5,0.866025,1.0,6.123234e-17,-0.433884,-0.900969,53,2021,0,0,0,0,0,0,1,0


### 5. Weryfikacja zbioru
Suma zamówionych ilości dla każdego klienta (restauracji) w całym okresie oraz podstawowe informacje o zbiorze: liczba wierszy, unikalnych dat, klientów i podkategorii oraz zakres dat.

In [15]:
sumIL = df.groupby('Klient')['Ilość'].sum().sort_values(ascending=False)
print("Suma zamawionych ilości produktów przez 5 lat dla każdej restauracji")
print(sumIL)

print("Wiersze:", df.shape[0])
print("Unikalne daty:", df['data_sprzedaży'].nunique())
print("Klienci:", df['Klient'].nunique())
print("Podkategorie:", df['Podkategoria'].nunique())

print("Min data:", df['data_sprzedaży'].min())
print("Max data:", df['data_sprzedaży'].max())
print("Liczba unikalnych dat:", df['data_sprzedaży'].nunique())

Suma zamawionych ilości produktów przez 5 lat dla każdej restauracji
Klient
R3    554666.31
R4    105255.12
R5     30193.47
R1     17318.90
R2      9983.28
Name: Ilość, dtype: float64
Wiersze: 465630
Unikalne daty: 1826
Klienci: 5
Podkategorie: 51
Min data: 2021-01-01 00:00:00
Max data: 2025-12-31 00:00:00
Liczba unikalnych dat: 1826


### 6. Inżynieria cech okazji (Event Features)
Dodanie specyficznych dni zwiększonego popytu w restauracjach:
* **`czy_okazja`**: oznacza dni okazji generujących ruch w restauracjach: Dzień Babci (21.01), Dzień Dziadka (22.01), Walentynki (14.02), Dzień Kobiet (8.03), Dzień Matki (26.05), Dzień Ojca (23.06), Dzień Chłopaka (30.09), Halloween (31.10) i Andrzejki (30.11).
* **`czy_przed_okazja`**: tworzy 3-dniowe okno wyprzedzające (1–3 dni przed okazją), uwzględniając fakt, że restauracje zaopatrują się z wyprzedzeniem.

In [16]:
okazje = {
    "Walentynki":       (2, 14),
    "Dzień Kobiet":     (3, 8),
    "Dzień Matki":      (5, 26),
    "Dzień Ojca":       (6, 23),
    "Dzień Chłopaka":   (9, 30),
    "Dzień Babci":      (1, 21),
    "Dzień Dziadka":    (1, 22),
    "Andrzejki":        (11, 30),
    "Halloween":        (10, 31),
}

df['czy_okazja'] = 0
for nazwa, (m, d) in okazje.items():
    maska = (
        (df['data_sprzedaży'].dt.month == m) & 
        (df['data_sprzedaży'].dt.day == d)
    )
    df.loc[maska, 'czy_okazja'] = 1

# Opcjonalnie — okno 3 dni przed okazją (ludzie kupują wcześniej)
df['czy_przed_okazja'] = 0

for nazwa, (m, d) in okazje.items():
    target_dates = df['data_sprzedaży'].dt.normalize().apply(
        lambda x: pd.Timestamp(year=x.year, month=m, day=d)
    )

    diff = (target_dates - df['data_sprzedaży']).dt.days

    df.loc[(diff > 0) & (diff <= 3), 'czy_przed_okazja'] = 1

### 7. Sezonowość i zmienne opóźnione (Lags)
* **Szczyt sezonu:** Dodana zostaje flaga (`czy_szczyt_restauracja`) dla miesięcy najwyższej sprzedaży w gastronomii: maj–czerwiec oraz sierpień–październik.
* **Szczyt tygodniowy:** Utworzona zostaje flaga dla czwartków (`czy_czwartek`), które prawdopodobnie są kluczowym dniem zaopatrzenia przed weekendem.
* **Autokorelacja:** Na samym końcu generowana jest kluczowa cecha modelu: **`lag_364`**. Przesunięcie o dokładnie 364 dni (52 pełne tygodnie) pozwala modelowi oprzeć predykcję na wolumenie sprzedaży z dokładnie tego samego dnia tygodnia w roku poprzednim. Lag liczony jest osobno dla każdej pary klient × podkategoria; dla pierwszego roku danych (2021) nie ma roku poprzedniego, więc wartość jest pusta (`NaN`).

Na koniec gotowy zbiór zapisywany jest do `../dane/przetworzone/Dane_restauracje.csv`.

In [ ]:
df['czy_szczyt_restauracja'] = 0
mask_rest = df['data_sprzedaży'].dt.month.isin([5,6,8,9,10])  
df.loc[mask_rest, 'czy_szczyt_restauracja'] = 1
df['czy_czwartek'] = (df['data_sprzedaży'].dt.dayofweek == 3).astype(int)

In [18]:
# LAG
df = df.sort_values(['Klient', 'Podkategoria', 'data_sprzedaży'])
group = df.groupby(['Klient', 'Podkategoria'])['Ilość']
df['lag_364'] = group.shift(364)
df = df.sort_values(["data_sprzedaży","Podkategoria"])

In [13]:
df.to_csv(
    '../dane/przetworzone/Dane_restauracje.csv', 
    encoding='utf-8-sig', 
    sep=';',       # Rozdzielacz kolumn (średnik)
    decimal=',',   # Zamienia kropki na przecinki w liczbach
    index=False     
)